> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 12 · THE BUSINESS</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Unit economics, build vs buy, and benchmarking your own data</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Cost models · Samvaad comparison · self-hosting crossover · measure accuracy on YOUR audio</div>
</div>

**Time:** 50 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹3 &nbsp;·&nbsp; **Prereq:** Labs 02, 03, 05, 09, 10, 11

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## Why this is a lab and not a slide

Anybody can quote a price list. The skill is turning it into a **decision** — build or
buy, managed or self-hosted, v2 or v3, what to charge. All of that is arithmetic you
can run, and this notebook is the calculator.

---
## 1 · A complete unit-economics model

In [3]:
RATES = {
    "stt_hr": 30.0, "stt_diar_hr": 45.0,
    "tts_v2_10k": 15.0, "tts_v3_10k": 30.0,
    "xlate_10k": 20.0, "lid_10k": 3.50,
    "llm_in_1m": 29.28, "llm_cached_1m": 10.98, "llm_out_1m": 73.20,
    "doc_page": 0.50, "samvaad_min": 3.50,
}

def voice_call(minutes=3.0, caller_share=.40, chars_per_min=900,
               ctx_tokens=20_000, out_tokens=1_500, cached_share=0.0,
               tts_v3=False, diarize=False, telephony_min=0.60):
    stt = (RATES["stt_diar_hr"] if diarize else RATES["stt_hr"])/3600 * minutes*60*caller_share
    tts = (RATES["tts_v3_10k"] if tts_v3 else RATES["tts_v2_10k"])/10_000 * \
          minutes*(1-caller_share)*chars_per_min
    lin = ctx_tokens*((1-cached_share)*RATES["llm_in_1m"] + cached_share*RATES["llm_cached_1m"])/1e6
    lou = out_tokens*RATES["llm_out_1m"]/1e6
    tel = telephony_min*minutes
    return dict(STT=stt, TTS=tts, LLM=lin+lou, Telephony=tel,
                TOTAL=stt+tts+lin+lou+tel)

def doc_pipeline(pages, llm_validate=True, tokens_per_page=800):
    d = pages*RATES["doc_page"]
    l = pages*tokens_per_page*RATES["llm_in_1m"]/1e6 if llm_validate else 0
    return dict(DocAI=d, LLM=l, TOTAL=d+l)

import pprint; pprint.pp(voice_call(tts_v3=True))

{'STT': 0.6000000000000001,
 'TTS': 4.86,
 'LLM': 0.6954,
 'Telephony': 1.7999999999999998,
 'TOTAL': 7.955400000000001}


In [4]:
# Margin model — the four numbers a founder actually needs
def margin(cost_per_unit, price_per_unit, volume_per_month, fixed_monthly=0):
    rev = price_per_unit*volume_per_month
    cogs = cost_per_unit*volume_per_month
    gp = rev - cogs
    return {
        "revenue":      rev,
        "cogs":         cogs,
        "gross_profit": gp - fixed_monthly,
        "gross_margin": (gp - fixed_monthly)/rev if rev else 0,
        "breakeven_vol": fixed_monthly/(price_per_unit-cost_per_unit)
                         if price_per_unit > cost_per_unit else float("inf"),
    }

c = voice_call(tts_v3=False, cached_share=0.8)["TOTAL"]
m = margin(cost_per_unit=c, price_per_unit=15.0,
           volume_per_month=100_000, fixed_monthly=250_000)
print(f"cost/call     ₹{c:.2f}")
for k, v in m.items():
    print(f"{k:<14} {v:>14,.2f}" if isinstance(v, float) and k != "gross_margin"
          else f"{k:<14} {v:>14.1%}" if k == "gross_margin" else f"{k:<14} {v:>14,.0f}")

cost/call     ₹5.23
revenue          1,500,000.00
cogs               523,260.00
gross_profit       726,740.00
gross_margin            48.4%
breakeven_vol       25,595.35


---
## 2 · Build vs buy — Sarvam's own price is your benchmark

Samvaad went generally available at **₹3.50/minute**. Compare like for like.

> ⚠️ **Check before you quote this.** It is not documented whether ₹3.50/min bundles
> telephony. The comparison below shows both readings — verify with Sarvam before you
> put either number in a customer deck.

In [5]:
mins = 3.0
naive     = voice_call(minutes=mins, tts_v3=True)["TOTAL"]
optimised = voice_call(minutes=mins, tts_v3=False, cached_share=0.8)["TOTAL"]
tel       = 0.60 * mins
samvaad   = RATES["samvaad_min"] * mins

print(f"{'':<34}{'per call':>10}{'per min':>10}")
print("─" * 54)
print(f"{'DIY naive (v3, no cache), all-in':<34}₹{naive:>9.2f}₹{naive/mins:>9.2f}")
print(f"{'DIY naive, excl. telephony':<34}₹{naive-tel:>9.2f}₹{(naive-tel)/mins:>9.2f}")
print(f"{'DIY optimised, all-in':<34}₹{optimised:>9.2f}₹{optimised/mins:>9.2f}")
print(f"{'DIY optimised, excl. telephony':<34}₹{optimised-tel:>9.2f}₹{(optimised-tel)/mins:>9.2f}")
print(f"{'Samvaad (managed)':<34}₹{samvaad:>9.2f}₹{RATES['samvaad_min']:>9.2f}")
print("─" * 54)
print(f"\nManaged premium vs naive build     : {samvaad/(naive-tel)-1:>6.0%}")
print(f"Managed premium vs optimised build : {samvaad/(optimised-tel)-1:>6.0%}")

                                    per call   per min
──────────────────────────────────────────────────────
DIY naive (v3, no cache), all-in  ₹     7.96₹     2.65
DIY naive, excl. telephony        ₹     6.16₹     2.05
DIY optimised, all-in             ₹     5.23₹     1.74
DIY optimised, excl. telephony    ₹     3.43₹     1.14
Samvaad (managed)                 ₹    10.50₹     3.50
──────────────────────────────────────────────────────

Managed premium vs naive build     :    71%
Managed premium vs optimised build :   206%


**Two things follow.**

1. **Your cost model is sane** — a commercial managed product prices in the same
   order of magnitude, which is the sanity check you want before quoting anyone.
2. **The managed premium is large**, and it grows the more you optimise. That premium
   is exactly what a developer sells: you capture it by owning the plumbing.

The premium is not Sarvam overcharging. It is the real cost of orchestration, state,
retries, barge-in, observability, uptime and someone answering the phone at 3am. The
question is whether you want to own that work — and below a certain volume, you
emphatically do not.

In [6]:
# At what volume does the engineering to build it actually pay for itself?
ENGINEER_MONTHLY = 250_000          # one senior engineer, fully loaded
saving_per_call  = samvaad - (optimised - tel)

for vol in [1_000, 10_000, 50_000, 100_000, 500_000]:
    saved = saving_per_call*vol
    verdict = "BUILD" if saved > ENGINEER_MONTHLY else "BUY"
    print(f"{vol:>8,} calls/mo  saving ₹{saved:>11,.0f}  "
          f"vs ₹{ENGINEER_MONTHLY:,} eng cost → {verdict}")

breakeven = ENGINEER_MONTHLY/saving_per_call
print(f"\nBreakeven ≈ {breakeven:,.0f} calls/month.")
print("Below that, buying is not laziness — it is arithmetic.")

   1,000 calls/mo  saving ₹      7,067  vs ₹250,000 eng cost → BUY
  10,000 calls/mo  saving ₹     70,674  vs ₹250,000 eng cost → BUY
  50,000 calls/mo  saving ₹    353,370  vs ₹250,000 eng cost → BUILD
 100,000 calls/mo  saving ₹    706,740  vs ₹250,000 eng cost → BUILD
 500,000 calls/mo  saving ₹  3,533,700  vs ₹250,000 eng cost → BUILD

Breakeven ≈ 35,374 calls/month.
Below that, buying is not laziness — it is arithmetic.


### 2.1 · `ENGINEER_MONTHLY` was a single flat number. It shouldn't be.

Labs 09–11 were built after this one, and none of their findings were available when
`ENGINEER_MONTHLY = 250,000` was written above. Now that they exist, that single
number is worth splitting into three, because the evidence says "the cost of
building it yourself" is not one line item — it behaves differently depending on
*which* piece of the system you mean.

- **The hot path does not get cheaper.** Lab 10 measured MCP against the direct SDK
  on exactly the STT call this voice loop makes, and its own conclusion — echoing
  Lab 09's "integration is a `base_url` swap, not a rewrite" — was still **"hot path
  → direct SDK."** Schema-token cost on every turn and tail-latency variance rule out
  MCP for a live call. `ENGINEER_MONTHLY` as a stand-in for "build the STT → LLM →
  TTS loop, checkpointing, barge-in, evals, guardrails" is **not reduced** by
  anything in 09–11 — it's the same build effort Lab 07/08 already priced in.
- **Everything *around* the hot path got much cheaper to build.** Lab 09 showed
  every major framework inherits Sarvam with a one-line `base_url` change, and
  Lab 10 showed 30 tools (STT, TTS, translate, vision, pronunciation, analytics)
  arrive with **zero custom wrapper code** via MCP. Post-call summarisation,
  transcript translation, document lookups, analytics — the "everything around it"
  bucket from Lab 10 §6 — used to mean writing and maintaining N separate SDK
  wrappers. It now means pointing an off-the-shelf agent at a tool list. That's a
  real, quantifiable reduction in adjacent build cost that the original model
  charged nothing for and also credited nothing for.
- **A cost the original model omitted entirely: keeping a coding assistant
  reliable on this SDK is ongoing work, not a one-time setup.** Lab 11 measured
  this directly — a bare LLM writes broken Sarvam code by default, and even a
  well-fetched two-page context still hallucinated a parameter that isn't real.
  Closing that gap took an authored, maintained Agent Skill — which is a
  recurring line item (someone updates it every time the SDK changes, the same
  way this notebook's own `RATES` dict needs re-verifying against
  `docs.sarvam.ai/api/getting-started/pricing`), not a sunk cost you pay once.

The cell below replaces the single `ENGINEER_MONTHLY` with these three components
and reruns the build-vs-buy table so the "BUILD" verdict reflects what actually
got cheaper, what didn't, and what the original model was silently omitting.

In [7]:
# ── Revised build cost: three components instead of one flat number ──────
HOT_PATH_ENGINEER_MONTHLY = 250_000   # unchanged — Lab 10 found nothing that shrinks this

# Adjacent work (post-call summary, transcript translation, doc lookups,
# analytics): Lab 09/10 collapsed this from "write N SDK wrappers" to
# "point an agent at a tool list." Model it as a small fraction of one
# engineer's time for config/maintenance — NOT zero, but no longer a build.
ADJACENT_HANDROLLED_SHARE = 0.30      # what this bucket cost before 09/10 existed
ADJACENT_WITH_TOOLING_SHARE = 0.05    # config + occasional glue, post Lab 09/10
adjacent_handrolled  = ADJACENT_HANDROLLED_SHARE   * HOT_PATH_ENGINEER_MONTHLY
adjacent_with_tooling = ADJACENT_WITH_TOOLING_SHARE * HOT_PATH_ENGINEER_MONTHLY

# Context/skill maintenance: Lab 11 showed this cost EXISTS and is ongoing —
# the original model above priced it at zero. Someone keeps the Agent Skill
# current as the SDK changes, the same way RATES here needs re-verifying.
CONTEXT_MAINTENANCE_SHARE = 0.08
context_maintenance = CONTEXT_MAINTENANCE_SHARE * HOT_PATH_ENGINEER_MONTHLY

original_flat_estimate = HOT_PATH_ENGINEER_MONTHLY          # what section 2 used
revised_estimate = (HOT_PATH_ENGINEER_MONTHLY
                     + adjacent_with_tooling
                     + context_maintenance)

print(f"{'component':<38}{'₹/month':>12}")
print("─" * 50)
print(f"{'hot path (unchanged by 09/10/11)':<38}₹{HOT_PATH_ENGINEER_MONTHLY:>11,.0f}")
print(f"{'adjacent work, hand-rolled (old world)':<38}₹{adjacent_handrolled:>11,.0f}  (not charged — see below)")
print(f"{'adjacent work, with Lab 09/10 tooling':<38}₹{adjacent_with_tooling:>11,.0f}")
print(f"{'context/skill maintenance (Lab 11)':<38}₹{context_maintenance:>11,.0f}  (was ₹0 in the original model)")
print("─" * 50)
print(f"{'ORIGINAL flat ENGINEER_MONTHLY':<38}₹{original_flat_estimate:>11,.0f}")
print(f"{'REVISED, evidence-based total':<38}₹{revised_estimate:>11,.0f}")

print(f"\nAdjacent-work saving from Lab 09/10 tooling: "
      f"₹{adjacent_handrolled - adjacent_with_tooling:,.0f}/month")
print(f"Previously-uncosted overhead from Lab 11: ₹{context_maintenance:,.0f}/month")

if revised_estimate > original_flat_estimate:
    print("\nNet effect: HIGHER true build cost than the original single number.")
    print("The framework/MCP savings on adjacent work are real, but smaller than the")
    print("context-maintenance cost the original model never charged for at all.")
else:
    print("\nNet effect: lower true build cost than the original single number.")

print("\nRevised BUILD-vs-BUY table (same saving_per_call as section 2, new engineer cost):")
for vol in [1_000, 10_000, 50_000, 100_000, 500_000]:
    saved = saving_per_call * vol
    verdict = "BUILD" if saved > revised_estimate else "BUY"
    print(f"{vol:>8,} calls/mo  saving ₹{saved:>11,.0f}  "
          f"vs ₹{revised_estimate:>10,.0f} revised eng cost → {verdict}")

breakeven_revised = revised_estimate / saving_per_call
print(f"\nRevised breakeven ≈ {breakeven_revised:,.0f} calls/month "
      f"(was {breakeven:,.0f} with the original flat estimate).")

component                                  ₹/month
──────────────────────────────────────────────────
hot path (unchanged by 09/10/11)      ₹    250,000
adjacent work, hand-rolled (old world)₹     75,000  (not charged — see below)
adjacent work, with Lab 09/10 tooling ₹     12,500
context/skill maintenance (Lab 11)    ₹     20,000  (was ₹0 in the original model)
──────────────────────────────────────────────────
ORIGINAL flat ENGINEER_MONTHLY        ₹    250,000
REVISED, evidence-based total         ₹    282,500

Adjacent-work saving from Lab 09/10 tooling: ₹62,500/month
Previously-uncosted overhead from Lab 11: ₹20,000/month

Net effect: HIGHER true build cost than the original single number.
The framework/MCP savings on adjacent work are real, but smaller than the
context-maintenance cost the original model never charged for at all.

Revised BUILD-vs-BUY table (same saving_per_call as section 2, new engineer cost):
   1,000 calls/mo  saving ₹      7,067  vs ₹   282,500 revised eng co

---
## 3 · The self-hosting crossover

The number that wins bank deals: *at what volume does a dedicated GPU endpoint beat
per-call pricing — while also keeping the data inside their VPC?*

In [8]:
# Indicative SageMaker GPU pricing — REPLACE with your region's real numbers
INSTANCE_PER_HOUR = 95.0            # ₹/hour, ml.g5.xlarge-class, on-demand
HOURS_PER_MONTH   = 730

def crossover(managed_rate_per_hour_audio, instance_hourly, hours=HOURS_PER_MONTH):
    monthly_instance = instance_hourly*hours
    audio_hours = monthly_instance/managed_rate_per_hour_audio
    return monthly_instance, audio_hours

inst_cost, audio_hrs = crossover(RATES["stt_hr"], INSTANCE_PER_HOUR)
print(f"Dedicated endpoint : ₹{inst_cost:>10,.0f} / month")
print(f"Managed equivalent : {audio_hrs:>10,.0f} audio-hours / month")
print(f"                   = {audio_hrs*60:>10,.0f} audio-minutes / month")
print(f"                   ≈ {audio_hrs*60/3:>10,.0f} three-minute calls / month")
print("\nAbove that volume, self-hosting is cheaper AND the audio never leaves the VPC.")
print("Below it, you are paying for idle GPU.")

Dedicated endpoint : ₹    69,350 / month
Managed equivalent :      2,312 audio-hours / month
                   =    138,700 audio-minutes / month
                   ≈     46,233 three-minute calls / month

Above that volume, self-hosting is cheaper AND the audio never leaves the VPC.
Below it, you are paying for idle GPU.


> **How to use this in a sales conversation.** A CISO asks "can our data stay inside
> our perimeter?" You say yes, and then: *"and above roughly N calls a month it is also
> cheaper — here is the model, put your own volumes in."* That is a materially
> different conversation from a feature checkbox.

---
## 4 · Benchmark accuracy on YOUR data

Never quote a vendor benchmark to a customer. Measure on their audio. This takes
twenty minutes and it is the artefact that closes pilots.

In [9]:
# Drop your own (audio, ground-truth) pairs here. 10 is enough to signal.
BENCH = [
    # ("data/real_call_01.wav", "मेरा EMI due date क्या है"),
]

def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+(r[i-1] != h[j-1]))
    return d[-1][-1]/max(len(r), 1)

def benchmark(pairs, **stt_kw):
    import wave
    rows = []
    for path, truth in pairs:
        with open(path, "rb") as f:
            r = client.speech_to_text.transcribe(file=f, model="saaras:v3", **stt_kw)
        with wave.open(str(path), "rb") as w:
            secs = w.getnframes()/w.getframerate()
        cost.stt(secs)
        rows.append({"file": Path(path).name, "wer": wer(truth, r.transcript),
                     "truth": truth, "hyp": r.transcript})
    if rows:
        mean = sum(x["wer"] for x in rows)/len(rows)
        for x in rows: print(f"{x['wer']:>6.1%}  {x['file']:<24} {x['hyp'][:50]}")
        print(f"\nMEAN WER: {mean:.1%}  over {len(rows)} files")
        return mean
    print("Add real audio to BENCH and re-run. This is the most valuable cell in the lab.")
    return None

benchmark(BENCH, language_code="hi-IN")

Add real audio to BENCH and re-run. This is the most valuable cell in the lab.


---
## 5 · Pricing models — and their traps

| Model | Fits | The trap |
|---|---|---|
| Per-minute / per-call | Voice agents | Your cost is also per-minute — thin, volume-dependent margin |
| Per-page / per-document | Doc pipelines | Buyer benchmarks against BPO rates. Race to the bottom |
| **Per-seat SaaS** | Internal tools, dashboards | **Decouples revenue from API cost — best margin structure** |
| Outcome-based | Collections, lead-gen | Highest capture, highest risk. Needs provable attribution |
| Platform + implementation | Enterprise, on-prem | Where the real money is in Indian enterprise |

In [10]:
def compare_pricing(monthly_calls=50_000, cost_per_call=None):
    c = cost_per_call or voice_call(tts_v3=False, cached_share=0.8)["TOTAL"]
    cogs = c*monthly_calls
    models = {
        "per-call @ ₹15":            15*monthly_calls,
        "per-minute @ ₹6":           6*3*monthly_calls,
        "per-seat @ ₹8k × 40 seats": 8_000*40,
        "outcome @ ₹120 × 4% conv":  120*monthly_calls*0.04,
        "platform ₹6L + ₹4/call":    600_000 + 4*monthly_calls,
    }
    print(f"COGS at {monthly_calls:,} calls: ₹{cogs:,.0f}  (₹{c:.2f}/call)\n")
    print(f"{'model':<28}{'revenue':>13}{'gross':>13}{'margin':>9}")
    print("─"*63)
    for k, rev in models.items():
        print(f"{k:<28}₹{rev:>12,.0f}₹{rev-cogs:>12,.0f}{(rev-cogs)/rev:>9.0%}")

compare_pricing()

COGS at 50,000 calls: ₹261,630  (₹5.23/call)

model                             revenue        gross   margin
───────────────────────────────────────────────────────────────
per-call @ ₹15              ₹     750,000₹     488,370      65%
per-minute @ ₹6             ₹     900,000₹     638,370      71%
per-seat @ ₹8k × 40 seats   ₹     320,000₹      58,370      18%
outcome @ ₹120 × 4% conv    ₹     240,000₹     -21,630      -9%
platform ₹6L + ₹4/call      ₹     800,000₹     538,370      67%


> **The rule.** Never price cost-plus when the customer's alternative is a human.
> Price against the incumbent — ₹25–60 a call, ₹3–8 a page, ₹25–40k/month per agent —
> hold 50–70% gross margin, and let the customer keep the rest of the saving. That is
> what makes a pilot an easy yes.

---
## 6 · Document pipeline economics

In [11]:
for pages in [10_000, 100_000, 1_000_000]:
    p = doc_pipeline(pages)
    bpo_lo, bpo_hi = pages*3, pages*8
    print(f"{pages:>10,} pages   your cost ₹{p['TOTAL']:>12,.0f}   "
          f"BPO ₹{bpo_lo:>12,.0f}–₹{bpo_hi:>12,.0f}   "
          f"you could charge ₹{pages*1.5:>12,.0f} at 1/2 BPO")

print(f"\nThroughput ceiling per API key: 10 jobs/min × 10 pages "
      f"= {100*60*24:,} pages/day.")

    10,000 pages   your cost ₹       5,234   BPO ₹      30,000–₹      80,000   you could charge ₹      15,000 at 1/2 BPO
   100,000 pages   your cost ₹      52,342   BPO ₹     300,000–₹     800,000   you could charge ₹     150,000 at 1/2 BPO
 1,000,000 pages   your cost ₹     523,424   BPO ₹   3,000,000–₹   8,000,000   you could charge ₹   1,500,000 at 1/2 BPO

Throughput ceiling per API key: 10 jobs/min × 10 pages = 144,000 pages/day.


---
## 7 · Your turn — build your own model

In [12]:
# ─────────────────────────────────────────────────────────────────────────
# EDIT THIS CELL. This is the deliverable of the whole lab series.
# ─────────────────────────────────────────────────────────────────────────
MY = {
    "who":              "NBFCs with a ₹5–50 crore loan book",
    "job":              "outbound EMI reminder and collection calls",
    "language":         "Hindi + Marathi",
    "incumbent":        "in-house tele-callers at ₹40/call fully loaded",
    "my_cost_per_unit": voice_call(tts_v3=False, cached_share=0.8)["TOTAL"],
    "my_price":         15.00,
    "volume_month":     50_000,
    "fixed_month":      250_000,
}

m = margin(MY["my_cost_per_unit"], MY["my_price"], MY["volume_month"], MY["fixed_month"])
print(f"I help {MY['who']}")
print(f"  do {MY['job']} in {MY['language']},")
print(f"  replacing {MY['incumbent']}.\n")
print(f"  my cost      ₹{MY['my_cost_per_unit']:.2f}/unit")
print(f"  my price     ₹{MY['my_price']:.2f}/unit")
print(f"  gross margin {m['gross_margin']:.0%}")
print(f"  breakeven    {m['breakeven_vol']:,.0f} units/month")
print(f"  monthly GP   ₹{m['gross_profit']:,.0f}")

undercut = 1 - MY["my_price"]/40
print(f"\n  customer saves {undercut:.0%} vs the incumbent — "
      f"and I keep {m['gross_margin']:.0%}. Both sides win.")

I help NBFCs with a ₹5–50 crore loan book
  do outbound EMI reminder and collection calls in Hindi + Marathi,
  replacing in-house tele-callers at ₹40/call fully loaded.

  my cost      ₹5.23/unit
  my price     ₹15.00/unit
  gross margin 32%
  breakeven    25,595 units/month
  monthly GP   ₹238,370

  customer saves 62% vs the incumbent — and I keep 32%. Both sides win.


In [13]:
cost.report()

TOTAL        ₹   0.0000
              (₹1000 free credit → ₹1000.00 left)
              Estimated from published rates; actual billing usually lower


0

---
## ✅ Checkpoint

- [ ] You can state your cost per transaction, derived not guessed
- [ ] You know the volume at which building beats buying Samvaad
- [ ] You know the volume at which self-hosting beats the managed API
- [ ] You filled in the `MY` cell with your own ICP and numbers

## 🧪 Try this

1. Run the model for **three** different verticals. Which has the best margin structure?
2. Add a human-review queue at ₹8/escalation and a 12% escalation rate. Does it still work?
3. Model a per-seat SaaS instead of per-call. How does the picture change at 500 seats?
4. Benchmark WER on 10 real recordings from your domain — then price with that number in hand.

---

## 🏁 You have finished the series

| Lab | What you built |
|---|---|
| 00 | Cost meter + the `content is None` trap |
| 01 | Every API in one script |
| 02 | Saaras — 5 modes, 3 paths, the 8 kHz cliff |
| 03 | Bulbul — voices, controls, streaming, TTFB |
| 04 | Language layer + auto-routing pipeline |
| 05 | Sarvam-105B — reasoning, tools, caching, streaming |
| 06 | Document AI — schemas, lifecycle, limits, queue |
| 07 | Agentic — state, checkpointing, evals, guardrails |
| 08 | Voice agent — latency budget, barge-in, telephony |
| 09 | Framework interop — LangChain, LangGraph, CrewAI, n8n |
| 10 | MCP at runtime — tool-choice accuracy, latency, token cost |
| 11 | Context engineering — markdown docs, llms.txt, Context7, Agent Skills |
| 12 | The business model |

**Next:** pick one workflow in your own domain, ship it to a URL or a phone number, and
show it to one person who would pay for it. Everything above is preparation for that.